# Practice Session 07: Connected components

*Introduction to Network Science* (2025/26), UPF

***Version 1***

In this session we will learn how to use [`networkx`](https://networkx.github.io/) to analyze connected components of a graph. 

We will use the [Star Wars graph](https://github.com/evelinag/StarWars-social-network/tree/master/networks). The dataset is contained in this input file that you will find in our [data](https://github.com/chatox/networks-science-course/tree/master/practicum/data) directory:
* ``starwars.graphml``: co-occurence of characters in scenes in the Star Wars saga in [GraphML](http://graphml.graphdrawing.org/) format.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random
import numpy as np
import statistics

# 1. The Star Wars graph

The following code just loads the *Star Wars* graph into variable `g`.

***Do not change the `INPUT_GRAPH_FILENAME` below!** Instead, place the file appropriately in the same directory and make sure you submit it in the zip file along with the notebook. It will be considered a mistake otherwise.*

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

INPUT_GRAPH_FILENAME = "starwars.graphml"

# Read the graph in GraphML format
g_in = nx.read_graphml(INPUT_GRAPH_FILENAME)

# Re-label the nodes so they use the 'name' as label
g_relabeled = nx.relabel.relabel_nodes(g_in, dict(g_in.nodes(data='name')))

# Convert the graph to undirected
g = g_relabeled.to_undirected()

In [ ]:
# LEAVE AS-IS (or modify slightly if you want) — we will reuse this function

def plot_graph(g):

    # Create a figure
    plt.figure(figsize=(12,8))

    # Layout the nodes using a spring model
    nx.draw_spring(
        g, 
        with_labels=True, font_size=10,     # Make labels, with label box defined below
        bbox=dict(facecolor="red", edgecolor='black', boxstyle='round,pad=0.1', alpha=0.7),
        node_size=0     # Hide nodes, leave only labels
    )

    # Display
    plt.show()

plot_graph(g)

<font size="+1" color="red">Replace this cell with your answer to the following: Is this a connected graph? Why or why not?</font>

Next, compute the maximum, mean and standard deviation of the degree of the nodes.

You can make a list out of `g.degree()` and then use functions in the [`statistics`](https://docs.python.org/3/library/statistics.html) module (or [`numpy`](https://numpy.org/) package if you prefer).

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with code to compute the maximum, mean, and standard deviation of the degree of the nodes. Print the results rounded to only 1 decimal. </font>

<font size="+1" color="red">Replace this cell with your answer to the following: 

- Is this a scale-free network? 
- Why or why not? 
- What would be the expected value of the standard deviation of the degree for an ER network with the same average degree?

</font>




To check your intuitions, plot a histogram of the degrees in the network.

Make sure to:
- Include x and y axis labels.
- The bin width is small enough to see the structure but not so small as to be too sparse (it should not look like a comb).

*Tip: Matplotlib can help you with this.*

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your code to plot a histogram of the degree.</font>


# 2. Remove a fraction of edges

The following function `remove_edges_uniformly_at_random(g, p)` returns a new graph which is a copy of `g` in which a fraction `p` of edges have been removed (with uniform probability). Leave it as-is, we will use it below.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

def remove_edges_uniformly_at_random(g_in, p):
    # Check input is within bounds
    if p < 0.0 or p > 1.0:
        raise ValueError

    # Create a copy of the input graph
    g_out = g_in.copy()

    # Decide how many edges should be in the output graph
    target_num_edges = int((1.0-p) * g_in.number_of_edges())

    # While there are more edges than desired
    while g_out.number_of_edges() > target_num_edges:

        # Remove one random edge
        edge = random.choice(list(g_out.edges()))

        if g_out.has_edge(edge[0], edge[1]):
            g_out.remove_edge(edge[0], edge[1])

    # Return the resulting graph
    return g_out

Use the function above to create three graphs named `g15`, `g50`, and `g85` that should ***contain*** 15%, 50%, and 85% of the edges in the original graph. Then, plot those three graphs.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with code to create `g15`, `g50`, `g85` as described above, and to plot these three graphs. Use three different cells for the plots. </font>


<font size="+1" color="red">Replace this cell with a brief commentary of what you observe visually in these three graphs with respect to the connected components. As we remove more edges...

1. Is the number of **connected components** increasing or decreasing?
2. Is the **size of the largest connected component** increasing or decreasing?

</font>


The following function, `remove_edges_by_betweenness(g,p)`, removes a fraction `p` of the edges with the *highest* betweenness. Leave it as-is.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

def remove_edges_by_betweenness(g_in, p):
    # Check input is within bounds
    if p < 0.0 or p > 1.0:
        raise ValueError

    # Create a copy of the input graph
    g_out = g_in.copy()

    # Compute edge betweenness
    edge_betweenness = nx.edge_betweenness_centrality(g_out)
    edges_by_betweenness = sorted(edge_betweenness.items(), key=lambda x: x[1], reverse=True)

    # Decide how many edges should be in the output graph
    target_num_edges = int((1.0-p) * g_in.number_of_edges())

    # While there are more edges than desired
    while g_out.number_of_edges() > target_num_edges:
        edge_to_remove, betweenness = edges_by_betweenness.pop(0)
        g_out.remove_edge(*edge_to_remove)

    # Return the resulting graph
    return g_out

Next, we use this function to remove the top 50% of the edges by betweenness.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

g50b = remove_edges_by_betweenness(g, 0.50)
plot_graph(g50b)

<font size="+1" color="red">Replace this cell with a brief commentary on what you observe visually in this graph where the top 50% of edges by betweenness was removed, in comparison with the graph in which 50% of the edges were removed uniformly at random: 

- Which one has a larger number of connected components?
- Why does this happen?

</font>


# 3. Assigning connected components

Next, we will write some code to label each node with the connected component it belongs to. (We will use this to count the number of connected components later.)

This code will be structured around two functions: `assign_components` and `assign_component_recursive`. We give you the algorithm below, you just have to write the functions.

The function `assign_components(g)` takes as input a graph `g`, and returns a dictionary that maps every node in `g` to a *positive* integer indicating its connected component number (a unique integer "index" or "label" of each component). The function should do the following:

1. Create an empty dictionary `node2componentid`.
1. Start with `componentid = 1`.
1. Iterate through all the nodes in the graph:
    1. For each `node` that is not in the `node2componentid` dictionary, call `assign_component_recursive`, incrementing `componentid` by 1 in each call.
1. Return the `node2componentid` dictionary.

The function `assign_component_recursive(g, node2componentid, starting_node, componentid)` takes care of actually assigning component ids to nodes, and it does so by recursive calls to each neighbour of the starting node that has not been visited yet. It should take the following arguments:

1. A graph `g`
1. A dictionary `node2componentid`
1. A starting node `starting_node`
1. A number `componentid`

The function should do the following:

1. Set `node2componentid[starting_node] = componentid`, i.e. assign a component to the starting node.
1. For each neighbor of `starting_node`, if that neighbor is not in the `node2componentid` dictionary, call itself `assign_component_recursive(g, node2componentid, neighbor, componentid)` to assign a component id to neighbours.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your code for `assign_component_recursive` and `assign_components`, as described above.</font>


It is important that your implementation of these functions is correct, as you will need it in the following section. The cell below contains basic checks to ensure this. If everyting is OK, you should get no errors from the `assert` statements. If you get errors, revise the functions accordingly.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# Test on cycle graph
cycle_components = assign_components(nx.cycle_graph(10))
for componentid in cycle_components.values():
    assert componentid == 1, "All nodes in cycle graph should belong to the same component, with ID 1."

# Test on caveman graph
caveman_components = assign_components(nx.caveman_graph(3, 3))
for componentid in caveman_components.values():
    assert componentid in [1, 2, 3], "Nodes in caveman graph should belong to one of three components (1, 2, or 3)."
    assert len([n for n, c in caveman_components.items() if c == componentid]) == 3, "There should be exactly 3 nodes assigned to each component in the caveman graph."

In the cell below, we show the component assignment for the graph of edges removed by betweenness `g50b` of a few Star Wars characters: Luke Skywalker, Kylo Ren, Supreme Leader Snoke, and Jedi Master Plo Koon. They should all belong to different components, except for Luke and Kylo who should be in the same component.

*(If you have watched the movies, this should make sense to you.)*

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# Get component assignments of nodes using the function just defined
components_g50b = assign_components(g50b)

# Print component assignments
for character in ['LUKE', 'KYLO REN', 'SNOKE', 'PLO KOON'] :
    print(f"Character {character:<8} belongs to component {components_g50b[character]}")

# 4. Number of connected components

In this section, we will see how the number of connected components changes as we remove edges.

First, we will create a dictionary that summarizes the component sizes in the graph. Follow these steps:
1. Write a function `extract_connected_component_sizes(g)` that returns a dictionary with `componentid`s as keys and the size of components as values. Use the `assign_components` function that you just created.
2. Apply `extract_connected_component_sizes` to the graph `g50b` that we defined before (with 50% of nodes removed by betweenness) to create a dictionary `component_sizes_g50b`.
3. Print the id and size of all components that have a size **larger than 1**, **sorted** in descending order (from largest to smallest).

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your code to define the `extract_connected_component_sizes` function, create the `component_sizes_g50b` dictionary, and the list the largest components.</font>


Next, write a function `count_connected_components(g)` that returns the number of connected components in `g`. 

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your code for `count_connected_components(g)`.</font>

In the following cell, we show the number of components in the graph we looked at before. (It should be 21.)

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

print(f"Number of connected components in graph g50b: {count_connected_components(g50b)}")

Use the `count_connected_components` function to obtain data for creating a plot.

You need to compute the number of components in the graphs obtained when 0%, 2%, 4%, ..., 98% of the edges in the StarWars graph are removed, with the two methods we defined before.

Store this data in two dictionaries:

* `ncomponents_removing_uniformly_at_random[p]` should contain the number of components when removing a fraction `p` of edges uniformly at random
* `ncomponents_removing_by_betweenness[p]` should contain the number of components when removing a fraction `p` of edges by betweenness

*Tip: You might find it handy to define a list of fraction values (0.0 to 1.0) using `range` or `np.arange`, which you can use to loop over.*

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your code for computing the required data.</font>

Next, plot this data. In this plot, the x axis should be the ***fraction*** of removed nodes and the y axis the number of connected components. **Both axes should go from 0.0 to 1.0; remember to include both series, to include a legend, and to label the axes.**

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your code to generate the requested plot.</font>

<font size="+1" color="red">Replace this cell with a brief commentary of what you observe on this graph, answering the following questions:

1. Do you see a linear trend, or something else? 
2. When you remove 100% of the edges, what is the number of connected components as a fraction of the number of nodes?

</font>

If you run the code to calculate the number of connected components again, and plot the results, you will notice that the line corresponding to random removal is different every time. These fluctuations are, of course, a result of taking a random sample of edges to remove from the graph.

To obtain a better idea of the effect of random removal, we want to look at the *average* or *expected* number of connected components. We can do this by repeating the same experiment multiple times and computing the average number of components we obtain for a given fraction `p` over those experiments.

For the next task, do this by repeating the random removal experiment **50 times**, and plot the same graph as above but this time with the average. *Make sure to indicate that you are showing an an average in your plot.*

The code you write might take some time to run, but in most setups it should execute in *less than a minute*. If it's taking much longer, then probably the code is not correct. (It is absolutely possible to do this using pure Python, although if you know how to use `numpy` arrays you can make it faster!)

*Tip: reuse as much of your code as you can be wrapping it inside functions!*

---

Good practice for writing such loops:
- Try with fewer runs first, to make sure results are correct.
- Then increase the number of runs, and see if you get results within a reasonable amount of time.
- If not, iterate on the first steps, checking and editing the algorithm as needed.

---

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="red">Replace this cell with your code to calculate the average number of connected components over 50 runs.</font>

<font size="+1" color="red">Replace this cell with your code to plot the results.</font>

<font size="+1" color="red">Briefly comment on the difference between this graph and the one your created before.</font>

# Deliver your code, you must (individually)

A .zip file containing:

* This notebook.
* The `starwars.graphml` file.

<font size="-1" color="gray">(This cell, when delivering, remove.)</font>

<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>